# Gemini API: Translate a public domain book

In this notebook, you will explore Gemini model as a translation tool, demonstrating how to prepare data, create effective prompts, and save results into a `.txt` file.

This approach significantly improves knowledge accessibility. Various sources provide open-source books. In this notebook, you will use [Project Gutenberg](https://www.gutenberg.org/) as a resource.

This platform provides access to a wide range of books available for free download in PDF, eBook, TXT formats and more.


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [1]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is one directory up from the notebook, hence we need to use `../` to go up one directory. If the `.env` file is in the same directory as the notebook, you can omit it altogether. 

```
│
├── .env
└── examples
    └── Translate_a_Public_Domain_Book.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [2]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [3]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Data preparation

You will translate a well-known book written by Arthur Conan Doyle from Polish (trans. Eugenia Żmijewska) to English about the detective Sherlock Holmes. Here are the titles in Polish and English:

- Polish Title: Tajemnica Baskerville'ów: dziwne przygody Sherlocka Holmes
- English Title: The Hound of the Baskervilles


In [4]:
const fs = require("fs") as typeof import("fs");
const path = require("path") as typeof import("path");

const downloadFile = async (url: string, filePath: string) => {
  const response = await fetch(url);
  if (!response.ok) {
    throw new Error(`Failed to download file: ${response.statusText}`);
  }
  fs.mkdirSync(path.dirname(filePath), { recursive: true });
  const buffer = await response.blob();
  const bufferData = Buffer.from(await buffer.arrayBuffer());
  fs.writeFileSync(filePath, bufferData);
};


In [5]:
const sherlockHolmesBookUrl = "https://www.gutenberg.org/cache/epub/34079/pg34079.txt";
const sherlockHolmesBookPath = path.join("../assets/examples", "sherlock_holmes.txt");
await downloadFile(sherlockHolmesBookUrl, sherlockHolmesBookPath);


In [6]:
const bookText = fs.readFileSync(sherlockHolmesBookPath, "utf-8");


Books contain all sorts of fictional or historical descriptions, some of them rather literal and might cause the model to stop from performing translation query. To prevent some of those exceptions users are able to change `safetySetting` from default to more open approach.


In [ ]:
import { SafetySetting } from "@google/genai";

const safetySettings: SafetySetting[] = [
  {
    category: google.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
    threshold: google.HarmBlockThreshold.BLOCK_NONE,
  },
  {
    category: google.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
    threshold: google.HarmBlockThreshold.BLOCK_NONE,
  },
  {
    category: google.HarmCategory.HARM_CATEGORY_HARASSMENT,
    threshold: google.HarmBlockThreshold.BLOCK_NONE,
  },
  {
    category: google.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
    threshold: google.HarmBlockThreshold.BLOCK_NONE,
  },
];


In [37]:
async function generateOutput(prompt: string): Promise<string> {
  const response = await ai.models.generateContent({
    model: MODEL_ID,
    contents: prompt,
    config: {
      safetySettings: safetySettings,
    },
  });
  return response.text ?? "";
}


Begin by counting how many tokens are in the book to know if you need to proceed with the next step (split text into smaller chunks).


## Token Information

LLMs are constrained by token limits. In order to stay within the limit of Gemini Flash, it's important to ensure the following:

- Input token limit should not exceed: 1,048,576 tokens
- Output token limit should not exceed: 8,192 tokens

Since input chunks of post-translation text will serve as outputs, it's advisable to split them into smaller segments.

As you will see in the example, there are over 1000 small chunks. Calling API for the token count of each one would take a long time. Because of this estimate 1 token is equal to 4 characters will be used.

To account for the possible miscalculation of tokens and output potentially being longer than input each chunk will use up to 4*5000 characters. Remember to adjust the maximum to your needs.

For more details on tokens in Gemini models, refer to the documentation [here](https://ai.google.dev/gemini-api/docs/models/gemini).


### Split text into chunks

Adjust the number of whitespace characters used for splitting based on the book you are working with.

Then, ensure to consider token limitations to receive complete responses from the model.


In [ ]:
const bookEndMarker = "END OF THE PROJECT GUTENBERG EBOOK TAJEMNICA BASKERVILLE'ÓW: DZIWNE PRZYGODY SHERLOCKA HOLMES";
const bookChunks = bookText.slice(0, bookText.indexOf(bookEndMarker)).split(/\r?\n\r?\n/);
const numChunks = bookChunks.length;
console.log(`Number of chunks: ${numChunks}`);


Number of chunks: 1802


In [27]:
const estimatedTokenCounts: number[] = bookChunks.map((chunk) => (chunk ? chunk.length / 4 : 0));
console.log("Estimated token counts for each chunk:", estimatedTokenCounts);


Estimated token counts for each chunk: [
  136.5, 16.25,  6.5,   7.25,  25.25,    4,    59, 26.25,     0,
  51.25,     0,    0,    150,      0, 10.5,    11,  7.75,   8.5,
  10.25,   9.5,  9.5,   7.75,   9.25,  9.5,  7.25,     0,  11.5,
      0,     1,    5,    139,   22.5, 37.5,  12.5, 74.25,  45.5,
   11.5,  24.5,  8.5,     46,  11.75,   32,  50.5,  48.5, 45.75,
  22.25, 38.75,   53,   6.75,   6.25, 3.75,    55, 19.75, 15.75,
   24.5, 45.75,  4.5,    155,  22.25,   73,   137, 92.75,  9.75,
  54.25,  38.5, 5.25, 121.75, 129.75,   38,  9.25,   3.5, 10.25,
  15.75, 11.25, 15.5,     28,   31.5, 16.5, 13.75, 10.25, 74.25,
     45, 66.75, 13.5,  28.25,  11.25, 43.5,    34,     0,  1.25,
    7.5, 14.75, 17.5,   6.25,    5.5,    4,  79.5,  16.5,     9,
  112.5,
  ... 1702 more items
]


Identify the largest number of tokens in a chunk to check if any exceed your token limitations. In this case, the chunks are very small, so this should not be an issue.

If you encounter this problem, consider splitting the text into even smaller chunks.


In [30]:
console.log(Math.max(...estimatedTokenCounts));


277


### Group smaller chunks together

As observed above, certain chunks contain very few tokens. While the LLM can translate these sentences accurately, providing larger context can enhance its understanding of the content.

Let's combine smaller chunks into larger ones for improved results.


In [31]:
function chunkGrouping(chunks: string[], tokenCounts: number[], maxLen = 5000): string[] {
  const groupedChunks: string[] = [];
  let currentGroup = "";
  let currentTokenSum = 0;

  // Process each chunk and group them based on token limits
  for (let i = 0; i < chunks.length; i++) {
    const chunk = chunks[i];
    const count = tokenCounts[i];

    // Skip chunks that exceed the max token limit
    if (count > maxLen) {
      continue;
    }

    // Add a new chunk if there is space available in the current group.
    if (currentTokenSum + 1 + count <= maxLen) {
      currentGroup += `\n\n${chunk}`;
      currentTokenSum += 1 + count; // Count in 1 token for newlines
    } else {
      // If adding this chunk exceeds the current group's capacity, start a new group
      groupedChunks.push(currentGroup);
      currentGroup = chunk;
      currentTokenSum = count;
    }
  }

  if (currentGroup) {
    // Add the last remaining group
    groupedChunks.push(currentGroup);
  }

  return groupedChunks;
}

const chunks = chunkGrouping(bookChunks, estimatedTokenCounts);
console.log(chunks.length);


13


The last step of splitting is checking if the non-estimated token counts are acceptable.


In [33]:
for (const chunk of chunks) {
  const tokenCount = await ai.models.countTokens({
    model: MODEL_ID,
    contents: chunk,
  });
  console.log(`Chunk token count: ${tokenCount.totalTokens}`);
}


Chunk token count: 6661
Chunk token count: 6750
Chunk token count: 6761
Chunk token count: 6655
Chunk token count: 6875
Chunk token count: 6749
Chunk token count: 6953
Chunk token count: 6798
Chunk token count: 6816
Chunk token count: 6842
Chunk token count: 6910
Chunk token count: 6825
Chunk token count: 3305


Despite using estimates, this split minimizes the calls necessary to translate the book well, and most chunks are of similar token counts to each other.


## Translation

Your book is ready for translations! The selected book has been divided into 13 larger chunks.


### Prompt

Specify the original language of your book and the language you would like it to be translated into.


In [34]:
function translations(text: string, sourceLanguage: string, targetLanguage: string) {
  return `
      As a professional book translator,
      translate the following book from ${sourceLanguage} into ${targetLanguage}.

      Book to Translate:
      ${text}
    `;
}


### Translate chunks

The translation time depends on the amount of text being processed. Our selected book was 278 KB and the entire translation took 15 minutes.

Please note that larger texts may take several hours to complete.


In [38]:
const display = tslab.newDisplay();
const results: string[] = [];
display.text("Progress: 0.00%");
for (let chunkIndex = 0; chunkIndex < chunks.length; chunkIndex++) {
  const translation = await generateOutput(translations(chunks[chunkIndex], "Polish", "English"));
  results.push(translation);
  display.text(`Progress: ${(((chunkIndex + 1) / chunks.length) * 100).toFixed(2)}%`);
}


Progress: 100.00%

All done, now you can print the chunk before and after translation to see the results.

You can see, that translation effectively maintains the narrative's novel quality, proper grammar and sentence structure.

Next step will be joining chunks together and saving the translated book.


In [39]:
tslab.display.markdown(chunks[1]);


-- A teraz, panie Holmes -- rzekł -- pokażę panu coś aktualniejszego.
Oto dziennik „Devon County Chronicle” z dnia 14-go Maja roku
bieżącego. Jest w nim opis faktów, odnoszących się do śmierci sir
Karola Baskerville, która zdarzyła się na kilka dni przed ową datą.

Mój przyjaciel nachylił się i począł słuchać uważniej. Doktor
Mortimer czytał:

„Niedawna i nagła śmierć sir Karola Baskerville, kandydata liberalnego
z hrabstwa Devon, przejęła całą naszą okolicę zdumieniem i trwogą.
Jakkolwiek sir Karol Baskerville mieszkał w Baskerville Hall od
niedawna, lecz jego uprzejmość, hojność i szlachetność, zjednały mu
szacunek tych wszystkich, którzy mieli z nim do czynienia.
W dzisiejszej epoce parweniuszów miło jest widzieć potomka
starożytnego a podupadłego rodu, odzyskującego fortunę swych
przodków i przywracającego świetność staremu nazwisku.

„Sir Karol, jak wiadomo, zyskał duży majątek na spekulacyach w Afryce
południowej, a będąc z natury przezornym, nie kusił dalej szczęścia,
które mogłoby się odwrócić od niego, jak to czyni z innymi,
i zrealizowawszy swoją fortunę, powrócił do Anglii.

„Przed dwoma laty zaledwie osiadł w Baskerville Hall; znane są jego
szerokie plany odnowienia starej siedziby i wprowadzenia ulepszeń
w gospodarstwie rolnem. Śmierć przeszkodziła ich urzeczywistnieniu.

„Będąc bezdzietnym, pragnął, aby cała okolica korzystała z jego
fortuny i niejeden z sąsiadów ma ważne powody do opłakiwania jego
przedwczesnego zgonu. Donosiliśmy często na tych szpaltach o jego
hojnych ofiarach na instytucye publiczne.

„Śledztwo nie wyjaśniło dotychczas okoliczności, towarzyszących
śmierci sir Karola. Krążą tu najdziwaczniejsze legendy, przypisujące
ów nagły zgon działaniu sił nadprzyrodzonych.

„Sir Karol był wdowcem, uważano go powszechnie za dziwaka. Pomimo
dużej fortuny, miał przyzwyczajenia proste, skromne. Personel służbowy
w Baskerville Hall składał się z kamerdynera, nazwiskiem Barrymore,
jego żony, kucharki i gospodyni w jednej osobie. Oboje zeznają, że
zdrowie sir Karola w ostatnich czasach było nietęgie, że rozwijała się
w nim choroba serca, objawiająca się bladością, brakiem tchu
i częstemi omdleniami. Doktor Jerzy Mortimer, przyjaciel i domowy lekarz
nieboszczyka, złożył zeznanie w tym samym duchu.

„Sir Karol Baskerville zwykł był co wieczór, przed pójściem na
spoczynek, przechadzać się po słynnej alei wiązów przed zamkiem.
Małżonkowie Barrymore opowiadają, że wieczorom 4-go maja, ich pan
oznajmił im, że nazajutrz wyrusza do Londynu i kazał pakować kuferek.

„Owego wieczoru wyszedł, jak zwykle, na przechadzkę, wśród której
wypalał cygaro.

„Z tej przechadzki już nie powrócił.

„O dwunastej Barrymore, widząc drzwi frontowe jeszcze otwarte,
zaniepokoił się, i wziąwszy latarnię, poszedł szukać swego pana.
W ciągu dnia deszcz padał, więc łatwo było dojrzeć ślady stóp sir
Karola wzdłuż alei. W połowie drogi jest furtka, prowadząca na łąkę.

„Były ślady, że sir Karol stał przy niej przez czas pewien,
następnie skręcił znowu w aleję. Znaleziono jego trupa na jej końcu.

„Pomiędzy innemi dotychczas nie wyjaśniono jednego faktu, wynikającego
z zeznań Barrymora, a mianowicie, że ślady kroków jego pana zmieniły
się z chwilą, gdy stał przy furtce i że odtąd szedł na palcach.

„Niejaki Murphy, cygan, handlujący końmi, znajdował się wówczas na
łące, w pewnej odległości od nieboszczyka, ale sam przyznaje, że był
pijany. Powiada, że słyszał krzyki, ale nie może określić, skąd
wychodziły.

„Na ciele sir Karola nie było śladów przemocy lub gwałtu, a chociaż
lekarz zeznaje, że wyraz twarzy nieboszczyka był tak zmieniony, że on,
doktor Mortimer, w pierwszej chwili poznać go nie mógł, rzeczoznawcy
tłómaczą to anewryzmem serca, przy którym zachodzą podobne objawy.

„Sekcya wykazała przedawnioną wadę serca, sędzia śledczy potwierdził
badanie lekarskie. Pożądanem jest, aby spadkobierca sir Karola mógł
objąć jaknajprędzej majątek i prowadzić dalej chlubną działalność,
przerwaną tak nagle i tak tragicznie.

„Gdyby prozaiczne orzeczenie sędziego śledczego i rzeczoznawców nie
położyło końca romantycznym historyom, krążącym na temat owej
śmierci, nie łatwo byłoby znaleźć właściciela Baskerville Hall.

„Domniemanym spadkobiercą jest pan Henryk Baskerville, syn młodszego
brata sir Karola. Ów młodzieniec przed kilku laty wyruszył do Ameryki.
Zarządzono poszukiwania. Dowie się on zapewne niebawem o zmianie swego
losu”.

       *       *       *       *       *

Doktor Mortimer złożył gazetę i wsunął ją do kieszeni, mówiąc:

-- To są fakty, powszechnie znane.

-- Dziękuję panu za zwrócenie mojej uwagi na sprawę istotnie 
ciekawą -- rzekł pan Holmes. -- Czytałem o niej, coprawda, w dziennikach, 
ale w owym czasie byłem zajęty kameami, które zniknęły z muzeum
Watykańskiego. Chciałem się przysłużyć Papieżowi i straciłem z oczu
to, co się jednocześnie działo w Anglii. Ten artykuł, powiadasz pan,
zawiera fakty, znane powszechnie?

-- Tak.

-- Zechciej więc pan uwiadomić mnie teraz o tem, czego nikt nie wie.

Sherlock Holmes oparł się o poręcz fotelu i znowu złożył ręce tak,
aby palce stykały się końcami.

-- Uczynię to -- rzekł doktor Mortimer, zdradzając coraz większe
wzburzenie. -- Powiem panu to, czego nie mówiłem jeszcze nikomu.
Zataiłem to przed sędzią śledczym z pobudek, które pan zapewne
zrozumie. Jako człowiek nauki, nie chciałem zdradzać się publicznie,
iż wierzę w przesądy. Dalej, rozumiałem, że gdyby utwierdziła się
wiara w nadprzyrodzone siły, rządzące w Baskerville Hall, nikt nie
zechciałby objąć starego zamku w posiadanie. Dla tych dwóch powodów
nie zeznałem przed sądem wszystkiego, co wiem -- albowiem nie
zdałoby się to na nic sędziemu śledczemu -- ale z panem będę
zupełnie szczery.

„Okolica jest mało zaludniona, rezydencye rzadkie, a siedziby
włościańskie rozrzucone na znacznej odległości od siebie.

„Widywałem często sir Karola Baskerville, który był spragniony
towarzystwa. Oprócz p. Frankland w Lafterhall i naturalisty,
p. Stapleton, w pobliżu niema ludzi inteligentnych. Sir Karol
był skryty, małomówny. Zbliżyliśmy się z powodu jego choroby,
przytem łączyły nas wspólne upodobania naukowe. Przywiózł wiele
ciekawych wiadomości z Afryki Południowej. Spędziliśmy niejeden
miły wieczór na rozprawach o anatomii porównawczej.

„W ciągu ostatnich paru miesięcy spostrzegłem, że system nerwowy sir
Karola był nadwerężony. Wziął tak dalece ową legendę do serca, że
choć co wieczór odbywał spacery, nic go nie mogło skłonić do
wejścia na łąkę po zachodzie słońca. Prześladowała go wciąż obawa
upiorów, i pytał mnie nieraz, czym nie widział jakiego dziwacznego
stworzenia i czym nie słyszał szczekania. Te pytania zadawał mi zawsze
głosem drżącym.

„Wraziła mi się w pamięć jedna z moich wizyt u niego przed trzema
tygodniami. Stał w sieni. Gdym schodził z wózka, spostrzegłem, że
patrzy po przez moje ramię ze strachem w oczach. Odwróciłem się
i spostrzegłem jakiś kształt dziwny, podobny do dużego, czarnego
cielęcia. Przeleciało to po za moim wózkiem.

„Sir Karol był tak wzburzony, że poszedłem szukać tego osobliwego
zwierzęcia. Ale nigdzie go nie było. Mój pacyent nie mógł się
uspokoić. Zostałem z nim przez cały wieczór i wtedy, tłómacząc swe
wzburzenie, opowiedział mi całą tę historyę.

„Wspominam o tym drobnym epizodzie, gdyż nabiera on znaczenia wobec
tragedyi, która potem nastąpiła; na razie nie przywiązywałem do tego
wagi i dziwiłem się wzburzeniu sir Karola.

„Miał jechać do Londynu z mojej porady. Wiedziałem, że jest chory
na serce i że ciągły niepokój źle wpływa na jego zdrowie. Sądziłem,
że parę miesięcy, spędzonych na rozrywkach, uwolni go od tych mar
i przywidzeń. Nasz wspólny przyjaciel, pan Stapleton, był tego samego
zdania. I oto wynikło nieszczęście.

„Zaraz po śmierci sir Karola, kamerdyner Barrymore wyprawił do mnie
grooma Perkinsa; przybyłem do Baskerville Hall w godzinę po
katastrofie. Zbadałem wszystkie fakty, przytoczone w śledztwie.
Szedłem śladami, pozostawionemi w Alei Wiązów, obejrzałem miejsce przy
furtce, gdzie sir Karol zatrzymał się, i zmiarkowałem, że od owego
miejsca szedł na palcach i że nie było śladu innych kroków, oprócz
późniejszych, Barrymora. Wreszcie obejrzałem starannie zwłoki, które
pozostały nietknięte do mego przybycia.

„Sir Karol leżał nawznak z rozciągniętemi rękoma, jego twarz była
tak zmieniona, że zaledwie mogłem ją poznać. Na ciele nie było
żadnych obrażeń. Ale Barrymore w toku śledztwa uczynił jedno zeznanie
fałszywe. Powiedział, że nie było żadnych śladów dokoła trupa. Nie
spostrzegł ich, ale ja dostrzegłem -- zupełnie świeże, w pobliżu.

-- Czy ślady kroków?

-- Tak.

-- Męzkie, czy kobiece?

Doktor Mortimer patrzył na nas przez chwilę dziwnemi oczyma, wreszcie
odparł szeptem:

-- Panie Holmes, były ślady kroków... psa... olbrzymiego.




III.

Zagadka.


Przyznaję, że gdym słuchał tego opowiadania, przebiegały po mnie
dreszcze. I doktor był żywo poruszony.

-- Widziałeś pan te ślady? -- spytałem.

-- Tak, na własne oczy; tak wyraźnie, jak teraz widzę pana.

-- I nic pan nie mówiłeś?

-- Po co?

-- Dlaczego nikt inny nie dostrzegł tych śladów?

-- Pozostały o jakie dwa łokcie od trupa; nie zwrócono na nie uwagi
i jabym ich nie zauważył, gdybym nie był znał tej legendy.

-- Dużo jest psów w okolicy.

-- Tak, ale to nie był zwykły pies. Powiedziałem już panom, że był
ogromny.

-- Czy nie zbliżył się do trupa?

-- Nie.

-- Czy noc była jasna?

-- Nie; pochmurna i wilgotna.

-- Ale deszcz nie padał?

-- Nie.

-- Jak wygląda aleja?

-- Jest wysadzana dwoma rzędami wiązów i opasana żywopłotem,
wysokości 12 stóp. Sama aleja ma 8 stóp szerokości.

-- Czy jest co pomiędzy żywopłotem a aleją?

-- Tak; po obu stronach biegnie trawnik, szerokości 6 stóp.

-- O ile zrozumiałem, jest wejście przez furtkę?

-- Tak; furtka prowadzi na łąkę.

-- Czy są inne wejścia?

-- Niema.

-- Tak, iż aby wkroczyć w Aleję Wiązów, trzeba wyjść z domu, lub
iść przez łąkę?

-- Jest trzecie wejście przez altanę, na drugim końcu alei.

-- Czy sir Karol doszedł do tej altany?

-- Nie; znaleziono go o pięćdziesiąt łokci od niej.

-- A teraz, zechciej mi pan powiedzieć, doktorze Mortimer -- będzie to
szczegół ważny -- czy ślady, któreś pan spostrzegł, pozostały na
żwirze, czy na trawie?

-- Nie mogłem dojrzeć śladów na trawie.

-- Czy ślady były po tej samej stronie, co furtka?

-- Tak, po tej samej.

-- Bardzo mnie pan zaciekawia. Jeszcze jedno pytanie. Czy furtka była
zamknięta?

-- Zamknięta na klucz i zaryglowana.

-- Jak jest wysoka?

-- Ma około 4-ch stóp.

-- A więc można ją łatwo przesadzić?

-- Tak.

-- A jakie ślady znalazłeś pan przy furtce?

-- Żadnych, któreby mogły zwrócić uwagę.

-- Czyżeś pan nie oglądał ziemi dokoła?

-- I owszem, oglądałem.

-- I nie dostrzegłeś pan żadnych śladów?

-- Były bardzo niewyraźne. Widocznie sir Karol stał tutaj przez pięć
do dziesięciu minut.

-- Skąd pan to wie?

-- Gdyż popiół z cygara spadł dwa razy na ziemię.

-- Wybornie. Znaleźliśmy kolegę wedle naszego serca. Prawda, Watson?
Ale jakież to były ślady?

-- Ślady stóp na żwirze. Innych znaków dostrzedz nie mogłem.

Sherlock Holmes uderzył ręką w kolano.

-- Czemu mnie tam nie było! -- zawołał. -- Jest to wypadek bardzo
ciekawy, dający obfite pole do naukowej ekspertyzy. Czemuż nie mogłem
odczytać tej żwirowej karty, zanim ją zatarły inne stopy! Doktorze
Mortimer, że też mnie pan nie uwiadomił wcześniej! Jeśli nam się nie
uda wyświetlić sprawy, cała odpowiedzialność spadnie na pana.

-- Nie mogłem pana wzywać, panie Holmes, bo w takim razie musiałbym
ujawnić te fakty przed całym światem, a mówiłem już, żem sobie tego
nie życzył. Zresztą... zresztą...

-- Czemu pan nie kończysz?

-- Bywają dziedziny, niedostępne nawet dla najbystrzejszych
i najdoświadczeńszych detektywów...

-- Chcesz pan powiedzieć, że wchodzi to w zakres rzeczy
nadprzyrodzonych?

-- Tego nie twierdzę stanowczo.

-- Ale pan tak myślisz w głębi ducha.

-- Od owej tragedyi doszły do mojej świadomości fakty, sprzeczne
z ogólnemi prawami natury.

-- Naprzykład?

-- Dowiaduję się, że przed tą katastrofą, kilku ludzi widziało na
łące niezwykłe stworzenie, podobne do złego ducha Baskervillów.
Wszyscy mówią, że był to pies ogromny, przezroczysty. Wypytywałem
tych ludzi -- jeden z nich jest włościaninem, drugi kowalem,
trzeci farmerem. Ich zeznania są jednakowe. W całej okolicy
zapanował strach przesądny. Nikt po nocy nie przejdzie przez łąkę.

-- I pan, człowiek nauki, wierzy w takie baśnie? -- zawołał Holmes.

-- Dotychczas moje badania ogarniały świat zmysłów; usiłowałem
walczyć z chorobą, ale ze złymi duchami walczyć nie umiem. Zresztą
musisz pan przyznać, że ślady stóp są dowodem materyalnym. Pies
Hugona nie był także zjawiskiem nadprzyrodzonem skoro mógł zagryźć
na śmierć, a jednak miał w sobie coś szatańskiego.

-- Widzę, że pan przeszedłeś do obozu spirytystów. Ale zechciej mi
powiedzieć jeszcze jedno, doktorze Mortimer. Jeżeli pan skłaniasz się
ku takim zapatrywaniom, dlaczego przyszedłeś zasięgnąć mojej rady?
Powiadasz pan jednym tchem, że nie warto przeprowadzać śledztwa
w sprawie zabójstwa sir Karola, a jednocześnie prosisz mnie, żebym się
tą sprawą zajął.

-- Nie prosiłem o to.

-- Więc w jaki sposób mogę panu dopomódz?

-- Radząc mi, co mam zrobić z sir Henrykiem Baskerville, który
przybywa na dworzec Waterloo -- doktor Mortimer spojrzał na zegarek --
za godzinę i kwadrans -- dokończył.

-- Czy on jest spadkobiercą?

-- Tak. Po śmierci sir Karola zasięgaliśmy wiadomości o tym
gentlemanie i dowiedzieliśmy się, że ma fermę w Kanadzie. Wedle
naszych informacyj, jest to młodzieniec bez zarzutu. Nie mówię teraz
jako lekarz, lecz jako wykonawca testamentu sir Karola.

-- Czy niema innych kandydatów do spadku?

-- Żadnego. Mógłby nim być tylko Roger Baskerville, najmłodszy
z trzech braci, z których sir Karol był najstarszym. Drugi brat, zmarły
przedwcześnie, był właśnie ojcem owego Henryka. Trzeci, Roger, był
synem marnotrawnym, żywą podobizną duchową starego Hugona. Tyle
nabroił w Anglii, że nie mógł już tu przebywać, uciekł do Ameryki
środkowej i umarł w roku 1876-ym na żółtą febrę. Henryk jest
ostatnim z Baskervillów. Za godzinę i pięć minut mam go spotkać
na dworcu Waterloo. Miałem depeszę z uwiadomieniem, iż przybył dziś
do Southampton. A teraz, panie Holmes, powiedz, jak mi radzisz postąpić?

-- Dlaczego sir Henryk nie miałby wrócić do domu swych ojców?

-- Wydaje się to rzeczą naturalną, a jednak, jeśli weźmiemy pod
uwagę, że każdego z Baskervillów, który tam przebywa, czeka śmierć
nagła i gwałtowna... Jestem pewien, że gdyby sir Karol mógł był ze
mną mówić przed katastrofą, byłby mi zakazał wprowadzać ostatniego
z rodu do owej przeklętej rezydencyi. Z drugiej strony, dobrobyt
całej okolicy zależy od przebywania dziedzica w Baskerville Hall.
Całe dzieło, zapoczątkowane przez sir Karola, pójdzie w niwecz,
jeśli nikt nie zamieszka na zamku. Boję się, aby dobro tej okolicy
nie skłoniło mnie do nielojalnego postąpienia z sir Henrykiem
i dlatego proszę pana o radę.

Holmes zastanawiał się długą chwilę.

-- Zatem -- odezwał się -- według pańskiego przekonania, jakaś siła
nieczysta grozi w tych stronach Baskervillom. Wszak pan w to wierzy
święcie?

-- Gotów jestem przypuszczać, że tak jest.

-- W takim razie, ta siła nieczysta może pastwić się nad Baskervillem
równie dobrze w Londynie, jak i w Devonshire. Szatan, którego
działanie byłoby umiejscowione, nie byłby groźnym.

-- Starasz się pan ośmieszyć moją obawę, panie Holmes. Lecz gdybyś
sam widział te rzeczy nadprzyrodzone, odechciałoby ci się żartów.
Z pańskich słów miarkuję, że ów młodzieniec jest równie bezpieczny
w Londynie, jak i w Devonshire. Przybywa za pięćdziesiąt minut. Cóż
mi pan radzisz?

-- Radzę wziąć dorożkę, zawołać psa, który skowyczy za drzwiami
i jechać na dworzec Waterloo na spotkanie sir Henryka Baskerville.

-- A potem?

-- A potem nic mu pan nie powiesz, dopóki nie namyślę się w tym
względzie.

-- Jak długo potrzebujesz pan namyślać się?

-- Dwadzieścia cztery godziny. Poproszę cię, doktorze Mortimer, abyś
mnie odwiedził jutro rano o dziesiątej. A zechciej przywieźć ze sobą
sir Henryka Baskerville. To mi ułatwi wykonanie mego planu.

-- Zrobię, jak pan chcesz.

Doktor zapisał godzinę na mankiecie i wyszedł. Pan Holmes zatrzymał go
na schodach.

-- Jeszcze jedno pytanie -- rzekł. -- Powiadasz pan, że przed śmiercią
sir Karola kilku ludzi widziało to zjawisko na łące?

-- Tak, trzech ludzi.

-- Czy który z nich widział je potem?

-- Nie wiem.

-- Dziękuję panu. Dowidzenia.

Holmes powrócił na swoje miejsce. Był widocznie zadowolony.

-- Wychodzisz, Watson? -- rzekł.

-- Czy potrzebujesz mojej pomocy?

-- Nie, mój drogi. Poproszę cię o nią dopiero w chwili działania.
Sprawa wyjątkowa. Przechodząc obok Bradleya, zechciej wstąpić do
sklepu i każ mi przynieść paczkę najmocniejszego tytoniu. Jeżeli
możesz, byłbym ci bardzo obowiązany, gdybyś tu nie wracał przed
wieczorem, a wtedy zestawimy nasze wrażenia i poglądy.

Wiedziałem, że samotność jest niezbędną mojemu przyjacielowi
w chwilach, gdy zastanawiał się nad poszlakami spraw kryminalnych, gdy
wyciągał wnioski i tworzył teorye, które okazywały się zawsze
słusznemi. To też cały dzień spędziłem w klubie i dopiero około
dziewiątej powróciłem do mieszkania przy Bakerstreet.

Gdy drzwi otworzyłem, zdało mi się, że w mieszkaniu był pożar;
światło lampy ukazywało się jakby za czarną mgłą. Po chwili
zmiarkowałem, że dym pochodzi nie od ognia, lecz od mocnego tytoniu.
Wśród kłębów ujrzałem Holmesa w szlafroku. Siedział w fotelu
z fajeczką w ustach. Na stole leżało kilka zwitków papieru.
Odkaszlnąłem.

-- Zaziębiłeś się? -- spytał.

-- Nie, ale można się tu udusić.

-- Otwórz okno. Widzę, że spędziłeś cały dzień w klubie...

-- Po czem to miarkujesz, Holmes?

-- Jesteś rzeźwy, pachnący, w dobrym humorze. Nigdy nie domyślisz się,
gdzie ja byłem.

-- Nie będę nad tem suszył głowy. Powiesz mi sam.

-- A więc byłem w Devonshire.

-- Myślą?

-- Tak. Moje ciało pozostało tutaj, na tym fotelu, i skonsumowało dwa
olbrzymie imbryki kawy i niezliczoną moc tytoniu. Po twojem wyjściu
posłałem do Stamforda po mapę tej okolicy i błądziłem po niej przez
dzień cały. Pochlebiam sobie, że każda piędź ziemi jest mi teraz
dobrze znana.

-- To zapewne mapa o wielkiej skali?

-- Tak.

Rozwinął ją i położył na kolanie.

-- Widzisz -- mówił -- oto łąka, a to -- Baskerville Hall.

-- Naokoło las.

-- Istotnie. A oto Aleja Wiązów, na lewo od łąki. Tu jest wioska
Grimpen, w której doktor Mortimer obrał swoją główną kwaterę.
W obrębie mil pięciu mało jest ludzkich siedzib. Oto Lafter Hall.
Tu -- domek przyrodnika Stapleton. Tutaj dwie formy: High Tore
i Fulmire. O czternaście mil dalej -- więzienie państwowe Princetown.
Dokoła i pośrodku -- łąki i trzęsawiska. Taki jest teren, na którym
rozegrała się owa tragedya. Postaramy się odtworzyć wszystkie jej
sceny i akty.

-- Miejscowość bezludna.

-- Tak. Dyabeł mógł się na niej rozgościć...

-- A zatem i pan skłaniasz się do nadprzyrodzonych wyjaśnień...

-- Sługami Dyabła mogą być ludzie z krwi i kości. Należy
przedewszystkiem rozstrzygnąć dwa zagadnienia: popierwsze, czy zaszła
zbrodnia? Powtóre: w jaki sposób ją spełniono? Jeżeli doktor Mortimer
nie jest w błędzie, jeśli okaże się, że mamy do czynienia
z nadprzyrodzonemi siłami, w takim razie dochodzenia sądowe są
bezużyteczne. Ale musimy wyczerpać wszelkie inne hypotezy, zanim
dojdziemy do tego wniosku. Możebyś zamknął okno. Znajduję, że
skoncentrowana atmosfera pomaga do skupienia myśli. Czy zastanawiałeś
się w ciągu dnia nad tą sprawą?

In [40]:
tslab.display.markdown(results[1]);


Here is the translation of the provided text from Polish into English:

***

“And now, Mr. Holmes,” he said, “I will show you something more recent. Here is the ‘Devon County Chronicle’ for May 14th of the current year. It contains an account of the facts relating to the death of Sir Charles Baskerville, which occurred a few days before that date.”

My friend leaned forward and began to listen more attentively. Dr. Mortimer read:

“The recent and sudden death of Sir Charles Baskerville, the Liberal candidate for the county of Devon, has filled our entire neighbourhood with astonishment and dismay. Although Sir Charles Baskerville had resided at Baskerville Hall for only a short time, his courtesy, generosity, and nobility of character had won him the respect of all who had dealings with him. In this age of parvenus, it is pleasant to see a descendant of an ancient but decayed family restoring the fortune of his ancestors and bringing lustre to an old name.

“Sir Charles, as is well known, amassed a large fortune through speculations in South Africa, and being by nature prudent, he did not tempt fortune further, as it might turn away from him, as it does with others. Having realised his fortune, he returned to England.

“He had settled at Baskerville Hall barely two years ago; his extensive plans for renovating the old mansion and introducing improvements in agriculture were well known. Death prevented their realisation.

“Being childless, he desired that the entire neighbourhood should benefit from his fortune, and not a few of his neighbours have ample reason to mourn his premature demise. We have frequently reported in these columns on his generous donations to public institutions.

“The inquest has not yet elucidated the circumstances attending Sir Charles’s death. The strangest legends circulate here, attributing his sudden demise to the action of supernatural forces.

“Sir Charles was a widower and was generally regarded as eccentric. Despite his great wealth, his habits were simple and modest. The household staff at Baskerville Hall consisted of a butler, named Barrymore, his wife, who served as cook and housekeeper combined. Both testify that Sir Charles’s health had been failing of late, and that a heart condition, manifesting in pallor, shortness of breath, and frequent fainting spells, had been developing in him. Dr. George Mortimer, a friend and the deceased’s personal physician, gave testimony to the same effect.

“Sir Charles Baskerville was accustomed to taking an evening stroll, before retiring, along the famous elm avenue in front of the mansion. The Barrymores state that on the evening of May 4th, their master informed them that he was departing for London the following day and instructed them to pack a valise.

“That evening he went out, as usual, for a walk, during which he was smoking a cigar.

“He never returned from that walk.

“At midnight, Barrymore, seeing the front door still open, became alarmed, and taking a lantern, went to look for his master. Rain had fallen during the day, so it was easy to discern Sir Charles’s footprints along the avenue. Halfway along, there is a gate leading to the meadow.

“There were indications that Sir Charles had stood by it for some time, then turned back into the avenue. His body was found at the end of it.

“Among other things, one fact arising from Barrymore’s testimony has not yet been explained: namely, that his master’s footsteps changed the moment he stood by the gate, and from then on, he walked on tiptoe.

“A certain Murphy, a Gypsy horse-dealer, was then on the meadow, at some distance from the deceased, but he himself admits that he was drunk. He states that he heard cries, but cannot specify whence they came.

“There were no marks of violence or struggle on Sir Charles’s body, and although the doctor testifies that the expression on the deceased’s face was so distorted that he, Dr. Mortimer, could not at first recognise him, experts attribute this to a cardiac aneurysm, which produces similar symptoms.

“An autopsy revealed an old-standing heart defect, and the coroner confirmed the medical findings. It is desirable that Sir Charles’s heir should take possession of the estate as soon as possible and continue the laudable work so suddenly and tragically interrupted.

“Had the prosaic verdict of the coroner and experts not put an end to the romantic stories circulating about this death, it would not have been easy to find an owner for Baskerville Hall.

“The presumptive heir is Mr. Henry Baskerville, son of Sir Charles’s younger brother. This young gentleman departed for America some years ago. Search efforts have been initiated. He will doubtless soon learn of the change in his fortunes.”

***

Dr. Mortimer folded the newspaper and tucked it into his pocket, saying:

“Those are the universally known facts.”

“I thank you for drawing my attention to a truly interesting matter,” said Mr. Holmes. “I had, it is true, read about it in the daily papers, but at that time I was engaged with the cameos that had disappeared from the Vatican Museum. I wished to render a service to the Pope and thus lost sight of what was simultaneously happening in England. This article, you say, contains universally known facts?”

“Yes.”

“Then kindly inform me now of what no one else knows.”

Sherlock Holmes leaned back against the armrest of his armchair and again put his hands together, fingertips touching.

“I shall do so,” said Dr. Mortimer, betraying increasing agitation. “I will tell you what I have not yet told anyone. I concealed it from the coroner for reasons which you will doubtless understand. As a man of science, I did not wish to publicly reveal that I believe in superstitions. Furthermore, I understood that if belief in the supernatural forces governing Baskerville Hall were to become firmly established, no one would wish to take possession of the old mansion. For these two reasons, I did not testify in court to everything I know—for it would have been of no use to the coroner—but with you, I shall be entirely frank.

“The district is sparsely populated, residences are rare, and farmhouses are scattered at considerable distances from one another.

“I frequently saw Sir Charles Baskerville, who was eager for company. Apart from Mr. Frankland of Lafterhall and the naturalist, Mr. Stapleton, there are no other intelligent people nearby. Sir Charles was reserved and taciturn. We grew close due to his illness, and moreover, we shared common scientific interests. He had brought many interesting facts from South Africa. We spent many pleasant evenings discussing comparative anatomy.

“Over the last few months, I had observed that Sir Charles’s nervous system was overtaxed. He had taken that legend so much to heart that although he took walks every evening, nothing could induce him to venture onto the moor after sunset. He was constantly haunted by the fear of spectres, and he would often ask me if I had seen any strange creature or heard any barking. These questions he always put to me in a trembling voice.

“One of my visits to him three weeks ago impressed itself upon my memory. He was standing in the hall. As I alighted from my trap, I noticed him looking over my shoulder with terror in his eyes. I turned and perceived a strange shape, resembling a large, black calf. It darted past my trap.

“Sir Charles was so agitated that I went to look for this peculiar animal. But it was nowhere to be found. My patient could not be calmed. I remained with him throughout the evening, and it was then, explaining his agitation, that he recounted the entire story to me.

“I mention this minor episode because it gains significance in light of the tragedy that followed; at the time, I attached no importance to it and wondered at Sir Charles’s agitation.

“He was to go to London on my advice. I knew he had a heart condition and that constant anxiety negatively affected his health. I believed that a few months spent in diversions would free him from these delusions and hallucinations. Our mutual friend, Mr. Stapleton, was of the same opinion. And then misfortune occurred.

“Immediately after Sir Charles’s death, the butler Barrymore dispatched the groom Perkins to me; I arrived at Baskerville Hall an hour after the catastrophe. I examined all the facts presented at the inquest. I followed the tracks left in the Elm Avenue, inspected the spot by the gate where Sir Charles had stopped, and perceived that from that point he had walked on tiptoe, and that there was no trace of other footsteps, apart from Barrymore’s later ones. Finally, I carefully examined the remains, which had remained untouched until my arrival.

“Sir Charles lay on his back with his arms outstretched, his face so altered that I could barely recognise it. There were no injuries on the body. But Barrymore made one false statement during the inquest. He said there were no tracks around the corpse. He did not observe them, but I did—quite fresh, nearby.

“Footprints?”

“Yes.”

“Male or female?”

Dr. Mortimer looked at us for a moment with strange eyes, then replied in a whisper:

“Mr. Holmes, there were footprints... of a dog... an enormous one.”

***

III. The Problem.

I confess that as I listened to this account, a thrill ran through me. And the doctor himself was visibly agitated.

“Did you see these tracks?” I asked.

“Yes, with my own eyes; as distinctly as I now see you.”

“And you said nothing?”

“Why should I?”

“Why did no one else perceive these tracks?”

“They were about two cubits from the corpse; they went unnoticed, and I myself would not have noticed them had I not been aware of the legend.”

“There are many dogs in the district.”

“Yes, but it was no ordinary dog. I have already told you, gentlemen, that it was enormous.”

“Did it not approach the corpse?”

“No.”

“Was the night clear?”

“No; cloudy and damp.”

“But it had not rained?”

“No.”

“What is the nature of the avenue?”

“It is bordered by two rows of elms and enclosed by a hedge, twelve feet high. The avenue itself is eight feet wide.”

“Is there anything between the hedge and the avenue?”

“Yes; on both sides runs a lawn, six feet wide.”

“As I understand it, there is an entrance through a gate?”

“Yes; the gate leads to the meadow.”

“Are there other entrances?”

“There are not.”

“So, to enter the Elm Avenue, one must leave the house or pass through the meadow?”

“There is a third entrance through a summer-house at the other end of the avenue.”

“Did Sir Charles reach this summer-house?”

“No; he was found about fifty cubits from it.”

“And now, please tell me, Dr. Mortimer—this will be an important detail—were the tracks you observed left on the gravel or on the grass?”

“I could not discern tracks on the grass.”

“Were the tracks on the same side as the gate?”

“Yes, the same side.”

“You intensely interest me. One more question. Was the gate locked?”

“Locked and bolted.”

“How high is it?”

“It is about four feet high.”

“So it could be easily surmounted?”

“Yes.”

“And what traces did you find by the gate?”

“None that would attract attention.”

“Did you not examine the ground around it?”

“Indeed, I did.”

“And you perceived no traces?”

“They were very faint. Apparently Sir Charles stood there for five to ten minutes.”

“How do you know that?”

“Because cigar ash had fallen twice upon the ground.”

“Excellent. We have found a kindred spirit. Have we not, Watson? But what exactly were these traces?”

“Footprints on the gravel. I could discern no other marks.”

Sherlock Holmes slapped his knee.

“Why was I not there!” he exclaimed. “This is a most interesting case, offering ample scope for scientific investigation. Why could I not have read that gravel card before other footsteps obliterated it! Dr. Mortimer, why did you not inform me sooner! If we fail to elucidate the matter, the entire responsibility will fall upon you.”

“I could not summon you, Mr. Holmes, for in that case I would have had to reveal these facts to the entire world, and I have already said that I did not wish to do so. Besides... besides...”

“Why do you not finish?”

“There are domains inaccessible even to the keenest and most experienced detectives...”

“Do you mean to say that this falls within the realm of the supernatural?”

“I do not assert that definitively.”

“But you believe so in your heart of hearts.”

“Since that tragedy, facts have come to my knowledge that are contrary to the general laws of nature.”

“For example?”

“I learn that before this catastrophe, several people saw an unusual creature on the moor, resembling the evil spirit of the Baskervilles. All say it was an enormous, transparent dog. I questioned these people—one of them is a peasant, another a blacksmith, the third a farmer. Their testimonies are identical. A superstitious fear has gripped the entire district. No one will cross the moor after nightfall.”

“And you, a man of science, believe in such fables?” exclaimed Holmes.

“Hitherto my investigations have encompassed the world of the senses; I have endeavoured to combat illness, but I know not how to fight evil spirits. Besides, you must admit that the footprints are material evidence. Hugo’s dog was also not a supernatural phenomenon, since it could bite to death, and yet it had something diabolical about it.”

“I see you have crossed over to the camp of the spiritualists. But tell me one more thing, Dr. Mortimer. If you incline towards such views, why have you come to seek my advice? You say in one breath that it is not worthwhile to investigate Sir Charles’s murder, and at the same time you ask me to take up the matter.”

“I did not ask that.”

“Then in what manner may I assist you?”

“By advising me what to do with Sir Henry Baskerville, who arrives at Waterloo Station—Dr. Mortimer glanced at his watch—in an hour and a quarter,” he finished.

“Is he the heir?”

“Yes. After Sir Charles’s death, we made inquiries concerning this gentleman and learned that he owns a farm in Canada. According to our information, he is a young man of irreproachable character. I speak now not as a physician, but as the executor of Sir Charles’s will.”

“Are there no other claimants to the inheritance?”

“None. The only other possible claimant would be Roger Baskerville, the youngest of the three brothers, of whom Sir Charles was the eldest. The second brother, who died prematurely, was in fact the father of this Henry. The third, Roger, was the prodigal son, a living spiritual image of old Hugo. He had caused so much trouble in England that he could no longer remain here, fled to Central America, and died of yellow fever in 1876. Henry is the last of the Baskervilles. I am to meet him at Waterloo Station in an hour and five minutes. I received a telegram informing me that he arrived in Southampton today. And now, Mr. Holmes, what do you advise me to do?”

“Why should not Sir Henry return to the home of his fathers?”

“It seems natural, and yet, if we consider that every Baskerville who resides there faces a sudden and violent death... I am certain that had Sir Charles been able to speak with me before the catastrophe, he would have forbidden me to introduce the last of the line to that cursed residence. On the other hand, the prosperity of the entire district depends on the heir residing at Baskerville Hall. The whole work initiated by Sir Charles will come to naught if no one inhabits the mansion. I fear that the welfare of this district might sway me to act disloyally towards Sir Henry, and that is why I seek your advice.”

Holmes pondered for a long moment.

“So,” he said, “according to your conviction, some unholy force threatens the Baskervilles in those parts. You truly believe this, do you not?”

“I am prepared to suppose that such is the case.”

“In that case, this unholy force can torment a Baskerville just as well in London as in Devonshire. A fiend whose operations were localised would not be so formidable.”

“You seek to ridicule my apprehension, Mr. Holmes. But had you yourself witnessed these supernatural occurrences, you would lose all desire for jesting. From your words, I gather that this young man is as safe in London as in Devonshire. He arrives in fifty minutes. What do you advise me?”

“I advise you to take a hansom, call for the dog that is whining behind the door, and proceed to Waterloo Station to meet Sir Henry Baskerville.”

“And then?”

“And then you will tell him nothing until I have deliberated upon the matter.”

“How long will you require for deliberation?”

“Twenty-four hours. I shall ask you, Dr. Mortimer, to call upon me tomorrow morning at ten o’clock. And please bring Sir Henry Baskerville with you. That will facilitate the execution of my plan.”

“I shall do as you wish.”

The doctor noted the hour on his cuff and departed. Mr. Holmes stopped him on the stairs.

“One more question,” he said. “You state that before Sir Charles’s death, several people saw this phenomenon on the moor?”

“Yes, three people.”

“Did any of them see it afterwards?”

“I do not know.”

“Thank you. Good day.”

Holmes returned to his seat. He was visibly satisfied.

“Are you going out, Watson?” he said.

“Do you require my assistance?”

“No, my dear fellow. I shall ask for it only when action is required. This is an exceptional case. As you pass Bradley’s, kindly step into the shop and have them send me a packet of the strongest tobacco. If you can, I should be much obliged if you did not return until evening, and then we shall compare our impressions and views.”

I knew that solitude was essential to my friend when he was pondering the clues of criminal cases, drawing conclusions, and forming theories that invariably proved correct. So I spent the entire day at my club, and it was not until about nine o’clock that I returned to our rooms in Baker Street.

When I opened the door, it seemed to me that there had been a fire in the room; the lamplight appeared as if through a black mist. After a moment, I perceived that the smoke emanated not from fire, but from strong tobacco. Amidst the clouds, I discerned Holmes in his dressing-gown. He was seated in an armchair with a pipe in his mouth. Several rolls of paper lay upon the table. I cleared my throat.

“Have you caught a cold?” he asked.

“No, but one could suffocate in here.”

“Open the window. I see you have spent the entire day at your club...”

“How do you gather that, Holmes?”

“You are refreshed, fragrant, and in good spirits. You will never guess where I have been.”

“I shall not rack my brains over it. You will tell me yourself.”

“Then I have been in Devonshire.”

“In thought?”

“Yes. My body has remained here, in this armchair, and has consumed two enormous pots of coffee and an immeasurable quantity of tobacco. After your departure, I sent to Stamford for a map of that district and have roamed over it all day long. I flatter myself that every inch of the ground is now well known to me.”

“That is presumably a large-scale map?”

“Yes.”

He unrolled it and laid it on his knee.

“You see,” he said, “here is the moor, and here—Baskerville Hall.”

“All around, woods.”

“Indeed. And here is the Elm Avenue, to the left of the moor. Here is the village of Grimpen, where Dr. Mortimer has established his headquarters. Within a radius of five miles, there are few human habitations. Here is Lafter Hall. Here—the naturalist Stapleton’s cottage. Here are two farms: High Tore and Fulmire. Fourteen miles further on—Princetown State Prison. All around and in the middle—moors and bogs. Such is the terrain upon which this tragedy unfolded. We shall endeavour to reconstruct all its scenes and acts.”

“A desolate locality.”

“Yes. The Devil himself might make his abode there...”

“And so you too incline towards supernatural explanations...”

“The Devil’s agents may be men of flesh and blood. First and foremost, two questions must be resolved: first, was a crime committed? Secondly: how was it perpetrated? If Dr. Mortimer is not mistaken, if it turns out that we are dealing with supernatural forces, then judicial inquiries are useless. But we must exhaust all other hypotheses before arriving at that conclusion. Perhaps you might close the window. I find that a concentrated atmosphere aids in the focusing of thought. Have you pondered this matter during the day?”

In [41]:
const translatedBookPath = path.join("../assets/examples", "sherlock_holmes_translated.txt");
fs.writeFileSync(translatedBookPath, results.join("\n\n"));
console.log(`Translated book saved to ${translatedBookPath}`);


Translated book saved to ../assets/examples/sherlock_holmes_translated.txt


## Next steps

Now you know how to properly prepare data, check token limitations and translate each chunk to obtain your results. Feel free to try this process with a book that interests you!
